<a href="https://colab.research.google.com/github/MaSBroEarl/-.-Text-classification/blob/main/distilbert_base_uncased.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Импортируем библиотеки

In [ ]:
!pip install torch torchvision transformers datasets accelerate scikit-learn scipy

In [ ]:
import pandas as pd
import numpy as np
import torch
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Проверка GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from transformers import (
    DistilBertTokenizer,          # Токенизатор для DistilBERT
    DistilBertForSequenceClassification,  # Модель для классификации
    Trainer,                       # Основной класс для обучения
    TrainingArguments,             # Параметры обучения
    EarlyStoppingCallback          # Ранняя остановка
)
from datasets import Dataset       # Формат данных для HuggingFace
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

Загружаем данные

In [ ]:
df = pd.read_csv('train.csv')

In [ ]:
# Удаляем ненужные колонки
df = df.drop(['id', 'location'], axis=1)

In [ ]:
# Заполняем пропуски в keyword
df['keyword'] = df['keyword'].fillna('')

In [ ]:
# Объединяем keyword с текстом
df['text_for_model'] = df['keyword'] + ' ' + df['text']

In [ ]:
# Удаляем дубликаты
before = len(df)
df = df.drop_duplicates(subset=['text'])

Препроцессинг

Для трансформеров  


НУЖНО Сделать:
   ✅ Токенизацию с помощью токенизатора модели
   ✅ Установка max_length (обрезание длинных текстов)
   ✅ Pad до одинаковой длины для batch
   ✅ Конвертация в формат PyTorch

In [ ]:
# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    df['text_for_model'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)


In [ ]:
# Dataset для HuggingFace
train_dataset = Dataset.from_dict({
    'text': X_train.tolist(),
    'label': y_train.tolist()
})

test_dataset = Dataset.from_dict({
    'text': X_test.tolist(),
    'label': y_test.tolist()
})

In [ ]:
#Загружаем токенизатор
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

In [ ]:
# Функция токенизации
def tokenize_function(examples):
    """
    Токенизация текста для трансформера.
    - padding='max_length': дополняем до максимальной длины
    - truncation=True: обрезаем длинные тексты
    - max_length=128: максимальная длина для твитов
    """
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors=None
    )

In [ ]:
# Применяем токенизацию
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

In [ ]:
# Удаляем оригинальный текст (экономия памяти)
train_dataset = train_dataset.remove_columns(['text'])
test_dataset = test_dataset.remove_columns(['text'])

In [ ]:
# Конвертируем в PyTorch формат
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

Выбираем предобученную модель:
**DistilBERT base model (uncased)**

Причины выбора:
1) быстрота (в 2x быстрее BERT)
2) компактность (250 MB)
3) почти такое же качество как и у BERT (97%)
4) хорошо работает с твитами

In [ ]:
# Загружаем модель
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2,
    id2label={0: 'not_disaster', 1: 'disaster'},
    label2id={'not_disaster': 0, 'disaster': 1}
)

In [ ]:
# Настройка обучения
# Функция для вычисления метрик
def compute_metrics(eval_pred):
    """
    Вычисление метрик качества.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, zero_division=0),
        'recall': recall_score(labels, predictions, zero_division=0),
        'f1': f1_score(labels, predictions, zero_division=0)
    }

In [ ]:
# Параметры обучения
training_args = TrainingArguments(
    # Основные параметры
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # Настройки обучения
    learning_rate=2e-5,
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    optim='adamw_torch',

    # Логирование и сохранение
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,

    # Выбор лучшей модели
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,

    # Дополнительно
    report_to=[],  # Отключаем логирование в сторонние сервисы
    fp16=False,    # Используем только если есть GPU
    seed=42,
)

print("✓ Параметры обучения настроены")
print(f"  Эпох: {training_args.num_train_epochs}")
print(f"  Батч: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

In [ ]:
#Обучение
#Создаем Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
# ============================================================
# FIX: ОБХОД ПРОБЛЕМЫ С TORCHVISION
# ============================================================
import sys
import torchvision

# Создаем заглушку для VideoReader, если её нет
if not hasattr(torchvision.io, 'VideoReader'):
    class VideoReaderStub:
        pass
    torchvision.io.VideoReader = VideoReaderStub
    print("✅ Patch applied for VideoReader")

In [ ]:
start_time = time.time()

trainer.train()

🎯 КЛЮЧЕВЫЕ ВЫВОДЫ:
   1. ✅ DistilBERT показывает лучшее качество по всем метрикам
   2. ✅ Модель стабильно улучшалась в процессе обучения
   3. ✅ Precision вырос значительно (+7.5%) - меньше ложных срабатываний
   4. ⚠️ Recall остался почти на том же уровне - можно улучшить

In [ ]:
print("""
📊 ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ:

┌─────────────────────────────┬──────────┬──────────┬──────────┬──────────┐
│ Модель                      │ Accuracy │ Precision│ Recall   │ F1-score │
├─────────────────────────────┼──────────┼──────────┼──────────┼──────────┤
│ 1. DistilBERT (fine-tuned)  │ 0.8441 🏆│ 0.8452 🏆│ 0.7766   │ 0.8094 🏆 │
│ 2. Эмбеддинги + LR          │ 0.8148   │ 0.7743   │ 0.8028 🏆│ 0.7883   │
│ 3. TF-IDF + LR              │ 0.8048   │ 0.7698   │ 0.7734   │ 0.7716   │
└─────────────────────────────┴──────────┴──────────┴──────────┴──────────┘

📈 УЛУЧШЕНИЕ DISTILBERT:
   • Accuracy:  +3.9% (по сравнению с TF-IDF)
   • Precision: +7.5% (по сравнению с TF-IDF)
   • F1-score:  +3.8% (по сравнению с TF-IDF)
   • Recall:    +0.3% (по сравнению с TF-IDF)
   """)